In [1]:
!git clone https://github.com/cherchin/ranking-masters-project.git

Cloning into 'ranking-masters-project'...
remote: Enumerating objects: 1776, done.
remote: Counting objects: 100% (1776/1776), done.
remote: Compressing objects: 100% (1443/1443), done.
remote: Total 1776 (delta 281), reused 1742 (delta 261), pack-reused 0 (from 0)
Receiving objects: 100% (1776/1776), 10.90 MiB | 16.32 MiB/s, done.
Resolving deltas: 100% (281/281), done.


In [2]:
import os

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["GEMINI_API_KEY"] = user_secrets.get_secret("GEMINI_API_KEY")
os.environ["GEMINI_MODEL"] = user_secrets.get_secret("GEMINI_MODEL")


In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [4]:
!pip install -q -U transformers accelerate sentencepiece tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 109.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 118.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.9/96.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69

In [5]:
%cd /kaggle/working/ranking-masters-project

/kaggle/working/ranking-masters-project


In [6]:

import os
import subprocess

script = "/kaggle/working/ranking-masters-project/src/llm/train-stargate.py"

processes = []

for batch, gpu in [(0, 0), (1, 1)]:
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu)

    log_file = open(f"/kaggle/working/batch_{batch + 1}.log", "w")

    process = subprocess.Popen(
        ["python", script, "--batch", str(batch)],
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )

    processes.append((process, log_file, batch))

for process, log_file, batch in processes:
    return_code = process.wait()
    log_file.close()
    print(f"Batch {batch + 1} finished with exit code {return_code}")

for batch in [1, 2]:
    print(f"\n===== BATCH {batch} =====")
    with open(f"/kaggle/working/batch_{batch}.log") as f:
        lines = f.readlines()
        print("".join(lines[-15:]))


Batch 1 finished with exit code 1
Batch 2 finished with exit code 1

===== BATCH 1 =====
Loading weights: 100%|██████████| 291/291 [00:46<00:00,  6.19it/s]
Model loaded successfully.

Initializing Gemini Oracle...
Traceback (most recent call last):
  File "/kaggle/working/ranking-masters-project/src/llm/train-stargate.py", line 357, in <module>
    main()
  File "/kaggle/working/ranking-masters-project/src/llm/train-stargate.py", line 285, in main
    oracle = Oracle()
             ^^^^^^^^
TypeError: Oracle.__init__() missing 1 required positional argument: 'model'


===== BATCH 2 =====
Loading weights: 100%|██████████| 291/291 [00:46<00:00,  6.19it/s]
Model loaded successfully.

Initializing Gemini Oracle...
Traceback (most recent call last):
  File "/kaggle/working/ranking-masters-project/src/llm/train-stargate.py", line 357, in <module>
    main()
  File "/kaggle/working/ranking-masters-project/src/llm/train-stargate.py", line 285, in main
    oracle = Oracle()
             ^^^^^^^